In [12]:
# =======================
# 1. SETUP
# =======================
!pip install torch torchvision torchaudio --quiet
!pip install segmentation_models_pytorch --quiet
!pip install grad-cam --quiet
!pip install nibabel --quiet
!pip install matplotlib --quiet
!pip install opencv-python --quiet

import os
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torchvision import transforms
from torch import nn
from torch.utils.data import DataLoader, Dataset
import nibabel as nib
from PIL import Image

# Create necessary folders
os.makedirs('./uploads', exist_ok=True)
os.makedirs('./outputs', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# =======================
# 2. UPLOAD MRI FILE
# =======================
from google.colab import files

print("Please upload your MRI image file (JPEG, PNG, or NIfTI NII/GZ format)")
uploaded = files.upload()

input_filename = next(iter(uploaded))
input_path = f'./uploads/{input_filename}'

Please upload your MRI image file (JPEG, PNG, or NIfTI NII/GZ format)


In [8]:
# =======================
# 3. LOAD PRETRAINED SEGMENTATION MODEL
# (simple U-Net from SMP for now)
# =======================
import segmentation_models_pytorch as smp

# Dummy pretrained U-Net (pretend it's trained on brain MRI)
# In production, replace with real trained weights

# Here we use an ImageNet backbone to fake it
model_seg = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,
).to(device)

# For now, randomly initialized (you would load your weights here)
model_seg.eval()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [11]:
# =======================
# 4. PREDICT SEGMENTATION MASK (CORRECTED)
# =======================
def load_image(image_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")

    if image_path.endswith(('.nii', '.nii.gz')):
        img = nib.load(image_path).get_fdata()
        img = np.squeeze(img)
        img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
        img = (img * 255).astype(np.uint8)
        if len(img.shape) == 2:
            img = cv2.merge([img, img, img])
    else:
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"cv2.imread failed. Unsupported image or corrupted: {image_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (256, 256))
    transform = transforms.ToTensor()
    img = transform(img)
    return img.unsqueeze(0)  # Add batch dimension

# Load image
input_img = load_image(input_path).to(device)

# Predict segmentation
with torch.no_grad():
    output_mask = torch.sigmoid(model_seg(input_img))
    output_mask = output_mask.squeeze().cpu().numpy()

# Threshold mask
binary_mask = (output_mask > 0.5).astype(np.uint8)

FileNotFoundError: Image file not found: ./uploads/aneurysm.jpeg

In [3]:
from nnunet.inference.predict import predict_from_folder

# Prepare folders
!mkdir -p /content/input_folder
!mkdir -p /content/output_folder

# Move uploaded file to input folder
!mv "{uploaded_filename}" /content/input_folder/

# Download pretrained model if needed (skipping here for now)
# You must later upload a pretrained nnU-Net model into /content/nnUNet_trained_models/

# Run prediction (fake command now, will replace later once model ready)
# predict_from_folder(
#     model_folder="/content/nnUNet_trained_models/nnUNet/3d_fullres/TaskXXX_Aneurysm/",
#     input_folder="/content/input_folder",
#     output_folder="/content/output_folder",
#     folds=[0],
#     save_npz=True,
#     num_threads_preprocessing=1,
#     num_threads_nifti_save=1,
#     mode="normal",
#     overwrite_existing=True,
#     all_in_gpu=False,
#     step_size=0.5,
#     checkpoint_name="model_final_checkpoint",
# )
print("⚠️ Prediction block is a placeholder until model is ready.")

ModuleNotFoundError: No module named 'nnunet'